In [1]:
import sys
import os

sys.path.append(os.path.abspath("../"))

import pandas as pd

from pipeline.version_config import VersionConfig
from pipeline.pipeline_run import PipelineRun
from pipeline.factory import PipelineFactory

## Config

**Feature engineering experiment notebook.**

This notebook retrains all models on the existing GCS data snapshot (`v3.4_real`) after
applying a feature engineering change (e.g. log-transforming count/velocity features).
No new data is pulled from BigQuery — the only variable changing between this run and
the v5.1 baseline is the feature engineering logic.

- **Data loaded from GCS:** `v3.4_real` (no BQ call, no data version bump)
- **Hyperparams:** reuses existing snapshot (`v1.1`) — no tuning
- **Model version:** minor bump (`v5.1 → v5.2`) — same data, new feature representation
- **Validation:** evaluated against the locked holdout (`splits/validation_ids.json`)

Set `dry_run = False` only when you're ready to write model artifacts and results to GCS.

In [ ]:
# Set False only when you're ready to write to GCS.
dry_run = True

config = (
    VersionConfig.load(use_synthetic=False)
    .snapshot_models()    # minor bump: v5.1 -> v5.2 (feature engineering, same data)
    .build()
)
run = PipelineRun(config)
stages = PipelineFactory.retrain_existing_data(config)

print('\nScenario:', stages.scenario)
print('data (read from GCS):', config.raw_version)
print('model version (write):', config.next_model_version)
print('dry_run:', dry_run)

---
## Stage 1 — DataLoader (GCS)

Reads the parquet snapshot at `config.raw_version`. **No BigQuery call.**

In [3]:
stages.loader.run(run)
run.summary()

Loaded snapshot 'v3.4_real': 60696 rows from 2026-04-29
  Polls: {'upload': 21398, '24h': 20906, '7d': 18392}
Loaded baselines 'v4.0': 28814 baseline videos, 974 median rows (974 channels)
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 2), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       None
  df_engineered  None
  df_train       None
  df_test        None
  df_val         None
  df_gen         None
  X_train        None
  X_test         None
  X_val          None
  X_val_unscaled  None
  X_gen          None
  y_train        None
  y_test         None
  y_val          None
  y_gen          None
  models         empty dict
  results        empty dict


## Stage 2 — DataPreprocessor

Pivots the long-format poll records into one row per video, joins channel baselines,
and runs structural cleanup. Writes `run.df_clean`.

In [4]:
stages.preprocessor.run(run)
run.summary()

Building clean dataset
snapshot cols: Index(['video_id', 'poll_timestamp', 'channel_id', 'channel_handle', 'title',
       'view_count', 'like_count', 'comment_count', 'face_count', 'brightness',
       'colorfulness', 'vertical', 'tier', 'description', 'tags',
       'duration_seconds', 'category_id', 'category_name', 'published_at',
       'poll_label', 'hours_since_publish', 'subscriber_count',
       'contains_synthetic_media'],
      dtype='object')

[1/3] Pivoting snapshots...
  Videos with all 3 polls: 18334 (dropped 3111 incomplete)
  Pivoted shape: (18403, 34)

[2/3] Joining baseline medians...
  Baseline join: 18403/18403 videos matched a channel median

[3/3] Cleaning data...
  Cleaned: 18403 rows × 40 columns

Clean dataset: 18403 rows × 40 columns
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 2), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFr

## Stage 3 — FeatureEngineer

Runs the full feature engineering chain on `run.df_clean`.

**This is where the experiment lives.** The log-transform changes are applied inside
`feature_engineering.py` — count/velocity features are passed through `np.log1p()`
before returning. Rates, ratios, binary flags, and categorical encodings are unchanged.
Writes `run.df_engineered`.

In [5]:
stages.engineer.run(run)
run.summary()

  all: dropped 414 rows with NaN in a baseline_median_* column or 0.0 baseline_median_engagement_rate
Engineering features

[1/9] Computing target variable...
  Target distribution: 55.1% above baseline, 44.9% below

[2/9] Computing velocity features...
  Computed velocity, upload momentum, normalized velocity, and acceleration features

[3/9] Computing ratio and baseline-normalized features...
  Computed ratio and baseline-normalized features

[4/9] Computing subscriber-normalized metrics...
  Computed subscriber-normalized metrics for upload/24h/7d

[5/9] Computing categorical features...
  Title categories:
title_category
neutral        11344
all_caps        2048
exclamation     1955
question        1572
listicle         562
how_to           230
clickbait        204
emoji_heavy       74
  Description categories:
desc_category
link_heavy        4729
minimal           4509
has_links         3837
has_hashtags      2697
neutral           1395
has_timestamps     774
long_form           4

## Stage 4 — DataSplitter

Loads locked validation IDs from GCS (`splits/validation_ids.json`).
Filters `run.df_engineered` to produce `df_val`, then stratifies the remaining
rows 80/20 into `df_train` / `df_test`.

In [6]:
stages.splitter.run(run)
run.summary()

DataSplitter — loaded holdout (5,193 val rows):
  df_val:    5,193 rows (28.9%)
  df_train: 10,236 rows (56.9%)
  df_test:   2,560 rows (14.2%)
DataSplitter — no generalization-vertical rows (Music/Sports) found.
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 2), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 87)
  df_train       populated  DataFrame shape=(10236, 87)
  df_test        populated  DataFrame shape=(2560, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  df_gen         populated  DataFrame shape=(0, 87)
  X_train        populated  DataFrame shape=(10236, 53)
  X_test         populated  DataFrame shape=(2560, 53)
  X_val          populated  DataFrame shap

## Stage 5 — Scaler

Fits `StandardScaler` on `X_train` (which now receives log-transformed inputs),
transforms `X_train`, `X_test`, and `X_val`. Captures `run.X_val_unscaled` so
Validator can apply each model's own historical scaler.

In [7]:
stages.scaler.run(run)
run.summary()

PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 2), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 87)
  df_train       populated  DataFrame shape=(10236, 87)
  df_test        populated  DataFrame shape=(2560, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  df_gen         populated  DataFrame shape=(0, 87)
  X_train        populated  DataFrame shape=(10236, 53)
  X_test         populated  DataFrame shape=(2560, 53)
  X_val          populated  DataFrame shape=(5193, 53)
  X_val_unscaled  populated  DataFrame shape=(5193, 53)
  X_gen          populated  DataFrame shape=(0, 53)
  y_train        populated  Series length=10236
  y_test         populated  Series length=25

## Stage 6 — Trainer

Trains LR, RF, XGB, and VotingClassifier using the existing hyperparams snapshot
(`v1.1`). No hyperparameter search — isolates the feature engineering change as the
sole variable vs. the v5.1 baseline.

In [8]:
stages.trainer.run(run)
run.summary()

Loaded hyperparams 'v1.1' (saved 2026-04-29)
  Models: ['LogisticRegression', 'RandomForest', 'XGBoost', 'VotingClassifier']
  Search: {'strategy': 'random', 'n_iter': 100, 'cv': 5, 'scoring': 'roc_auc'}
Loaded hyperparams from snapshot 'v1.1'.
  Note: injected l1_ratio=0.5 for elasticnet LR (missing from snapshot).
Training LogisticRegression (L1)...
Training RandomForestClassifier...
Training XGBClassifier...
Training VotingClassifier ensemble (RF + XGB, weights=[1, 2])...

=== ModelTrainer — test-set results ===
  lr_l1         AUC=0.7718  acc=0.7113  F1↑=0.7447
  rf            AUC=0.8659  acc=0.7844  F1↑=0.8108
  xgb           AUC=0.9096  acc=0.8285  F1↑=0.8468
  ensemble      AUC=0.9057  acc=0.8254  F1↑=0.8443
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 2), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populat

## Stage 7 — ModelSnapshotter

Saves trained model artifacts (model, scaler, feature_cols, metadata) to GCS
under `v5.2_*`.

In [ ]:
if not dry_run:
    stages.model_snapshotter.run(run)
    print('\nSaved models:', list(run.models.keys()))
else:
    print('[dry_run] skipping ModelSnapshotter')

## Stage 8 — Validator

Evaluates each model on the locked validation set using `X_val_unscaled` +
each model's own freshly-fitted scaler. Populates `run.results`.

In [12]:
stages.validator.run(run)
run.summary()


=== Validator — validation-set results ===
  lr_l1           AUC=0.7681  acc=0.7009  F1↑=0.7341
  rf              AUC=0.8723  acc=0.7957  F1↑=0.8228
  xgb             AUC=0.9104  acc=0.8342  F1↑=0.8521
  ensemble        AUC=0.9068  acc=0.8307  F1↑=0.8496
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 2), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 87)
  df_train       populated  DataFrame shape=(10236, 87)
  df_test        populated  DataFrame shape=(2560, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  df_gen         populated  DataFrame shape=(0, 87)
  X_train        populated  DataFrame shape=(10236, 53)
  X_test         populated  DataFrame shape=(2560, 53)

## Stage 9 — Persist Validation Results to GCS

In [ ]:
if not dry_run:
    stages.validation_results_snapshotter.run(run)
    # Appends to gs://maduros-dolce-capstone-data/models/{model_version}/validation_results.jsonl
else:
    print('[dry_run] skipping ValidationResultsSnapshotter')

Validation results appended → gs://maduros-dolce-capstone-data/models/v5.2/validation_results.jsonl
  lr_l1           AUC=0.7681  acc=0.7009  F1↑=0.7341
  rf              AUC=0.8723  acc=0.7957  F1↑=0.8228
  xgb             AUC=0.9104  acc=0.8342  F1↑=0.8521
  ensemble        AUC=0.9068  acc=0.8307  F1↑=0.8496


PipelineRun(model_version='v5.1', num_synth_rows=0, populated=['df_videos', 'df_baselines', 'df_medians', 'df_clean', 'df_engineered', 'df_train', 'df_test', 'df_val', 'df_gen', 'X_train', 'X_test', 'X_val', 'X_val_unscaled', 'X_gen', 'y_train', 'y_test', 'y_val', 'y_gen', 'models', 'results'])

---
## Results

Compare against v5.1 baseline on the locked holdout:

| Model | v5.1 AUC |
|-------|----------|
| lr_l1 | 0.761    |
| rf    | 0.872    |
| xgb   | 0.908    |

In [14]:
metric_cols = [
    'roc_auc', 'accuracy',
    'f1_above', 'precision_above', 'recall_above',
    'f1_below', 'precision_below', 'recall_below',
]

df_results = (
    pd.DataFrame(run.results).T
    [metric_cols]
    .astype(float)
    .round(4)
)
df_results.index.name = 'model'

df_results.style.highlight_max(color='#d4edda').format('{:.4f}')

,roc_auc,accuracy,f1_above,precision_above,recall_above,f1_below,precision_below,recall_below
model,,,,,,,,
lr_l1,0.7681,0.7009,0.7341,0.7251,0.7434,0.6583,0.6691,0.6479
rf,0.8723,0.7957,0.8228,0.7937,0.8540,0.7588,0.7986,0.7228
xgb,0.9104,0.8342,0.8521,0.8444,0.8599,0.8114,0.8209,0.8021
ensemble,0.9068,0.8307,0.8496,0.8386,0.8610,0.8064,0.8203,0.7930


In [15]:
# Top features per model — check whether log-transform shifted LR coefficient distribution
for name, entry in run.results.items():
    top = entry.get('top_features', [])[:5]
    print(f'\n{name}:')
    for f in top:
        key = 'coefficient' if 'coefficient' in f else 'importance'
        print(f'  {f["feature"]:35s}  {f[key]:+.4f}')


lr_l1:
  like_count_upload_vs_baseline        +12.9021
  view_count_upload_vs_baseline        -4.1040
  like_rate_24h                        +1.1244
  view_count_velocity_24h              -0.8141
  view_velocity_upload                 +0.5270

rf:
  like_rate_24h                        +0.1085
  view_velocity_ratio                  +0.0494
  like_count_upload_vs_baseline        +0.0408
  like_rate_upload                     +0.0396
  view_count_upload_vs_baseline        +0.0325

xgb:
  baseline_baseline_video_count        +0.0656
  like_rate_24h                        +0.0607
  tier_encoded                         +0.0384
  view_count_upload_vs_baseline        +0.0312
  like_count_upload_vs_baseline        +0.0311

ensemble:


---
## Final: GCS Write

Commit model version bump (`v5.1 → v5.2`) to GCS. Run only after all snapshots succeed.

In [ ]:
if not dry_run:
    config.commit()
else:
    print('[dry_run] skipping config.commit()')


Committed versions.json -> data v3.4, model v5.2, baselines v4.0, hyperparams v1.1
